<a href="https://colab.research.google.com/github/ashleyogorman/ICP_Milestone_1/blob/main/milestone_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ID Card OCR ICP Pipeline**


---



classifies which country an ID card belongs to

**Pipeline Stages**: image enhancement -> channel selection -> OCR



---


CS 8001 End to End Machine Learning

Ashley O'Gorman and Alexander Hardaway

# Setup
---

In [ ]:
import cv2
import numpy as np
import pytesseract
import easyocr
from PIL import Image
from datasets import load_dataset
import time

In [ ]:
print("--- Loading dataset 'lansinuote/ocr_id_card' ---")
try:
    ds = load_dataset("lansinuote/ocr_id_card", split='train')
    print("Dataset loaded successfully.")
    print(ds)
except Exception as e:
    print(f"Failed to load dataset: {e}")
    exit()

# Step 1: Image Enhancement Step


---


Contract: ***enhance_fn(image: np.ndarray) -> np.ndarray:***

The first step of the pipeline will be to use the cv2 library from python to enchance the base image if need be. The documentation to the library can be found here: https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html.

You may need to adjust the sharpness, brightness, or contrast of the image or make no changes at all. This is just a sample, as you are encouraged to find other helpful operations that can help increase the accuracy of the pipeline.

In [ ]:
# Baseline: no enhancement applied
def enhance_none(image: np.ndarray) -> np.ndarray:
    return image

Contrast

In [ ]:
# Ashley
def enhance_constrast_increase_25(image: np.ndarray) -> np.ndarray:
    contrast = 1.25
    brightness = 0
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

In [ ]:
# Ashley
def enhance_constrast_increase_50(image: np.ndarray) -> np.ndarray:
    contrast = 1.5
    brightness = 0
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

In [ ]:
# Ashley
def enhance_constrast_increase_75(image: np.ndarray) -> np.ndarray:
    contrast = 1.75
    brightness = 0
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

Brightness

In [ ]:
# Ashley
def enhance_brightness_decrease_50(image: np.ndarray) -> np.ndarray:
    contrast = 1.0
    brightness = -50
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

In [ ]:
# Ashley
def enhance_brightness_increase_50(image: np.ndarray) -> np.ndarray:
    contrast = 1.0
    brightness = 50
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

In [ ]:
# Ashley
def enhance_brightness_increase_100(image: np.ndarray) -> np.ndarray:
    contrast = 1.0
    brightness = 100
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

Laplacian

In [ ]:
# Ashley
def enhance_laplacian_simple(image: np.ndarray) -> np.ndarray:
    return cv2.Laplacian(image, -1)

In [ ]:
# Ashley
def enhance_laplacian_precise(image: np.ndarray) -> np.ndarray:
    laplacian_64f = cv2.Laplacian(image, cv2.CV_64F)
    laplacian_uint8 = np.uint8(np.absolute(laplacian_64f))
    return laplacian_uint8

In [ ]:
# Alex
def enhance_TODO(image: np.ndarray) -> np.ndarray:
  return

All enhancement options:

In [ ]:
enhancement_options = {
    "none": enhance_none,
    
    "+.25 contrast": enhance_constrast_increase_25,
    "+.50 contrast": enhance_constrast_increase_50,
    "+.75 contrast": enhance_constrast_increase_75,
    
    "-50 brightness": enhance_brightness_decrease_50,
    "+50 brightness": enhance_brightness_increase_50,
    "+100 brightness": enhance_brightness_increase_100,
    
    "simple laplacian": enhance_laplacian_simple,
    "precise laplacian": enhance_laplacian_precise,
    # TODO
}

# Step 2: Channel Selection Step
---
Contract: ***channel_fn(image: np.ndarray) -> np.ndarray:***

The next step is choose to extract a specific color channel from the image array. You may choose one color or a combination of colors or the orignal image as well. As before you can use the cv2 library to do these operations.

In [ ]:
# Baseline: no chanel applied
def channel_none(image: np.ndarray) -> np.ndarray:
    return image

Single Colors

In [ ]:
# Ashley
def channel_blue(image: np.ndarray) -> np.ndarray:
    b, g, r = cv2.split(image)
    return b

In [ ]:
# Ashley
def channel_green(image: np.ndarray) -> np.ndarray:
    b, g, r = cv2.split(image)
    return g

In [ ]:
# Ashley
def channel_red(image: np.ndarray) -> np.ndarray:
    b, g, r = cv2.split(image)
    return r

Multicolors

In [ ]:
# Alex
def channel_TODO(image: np.ndarray) -> np.ndarray:
  return

In [ ]:
channel_options = {
    "none": channel_none,

    "blue": channel_blue,
    "green": channel_green,
    "red": channel_red,
    
    # TODO
}

# Step 3: OCR Step

---
Contract: ***ocr_fn(image: np.ndarray) -> str:***

The final step is to actually perform actual ocr. For this step, we pass the processed numpy array of the input image to the ocr models. To do this, we provide two initial ocr models:

EasyOcr

Tesseract

In [ ]:
# Ashley
def ocr_easyocr(image: np.ndarray) -> str:
    result = easy_ocr_reader.readtext(image)
    return result

In [ ]:
# Alex
import re

import numpy as np
import pytesseract

# lang: chi_sim is Simplified Chinese. Its model also reads digits.
# psm: page segmentation mode, how Tesseract splits the image into text.
#   11 = sparse text, find as much text as possible in no particular order.
#   Scored best of psm 3, 4, 6, 11 and 12 on 20 sample cards.
TESSERACT_LANG = "chi_sim"
TESSERACT_PSM = 11

# Tesseract puts a space between Chinese characters ("方 泽 贤").
# This removes spaces that sit between two Chinese characters only,
# so numbers like "1990 8 28" keep their spaces.
_CJK_SPACE = re.compile(r"(?<=[\u4e00-\u9fff])\s+(?=[\u4e00-\u9fff])")


def ocr_tesseract(image: np.ndarray, psm: int = TESSERACT_PSM) -> list[dict]:
    """
    Returns one dict per line of text, in the same shape as the dataset's
    item['ocr'] minus 'cls' (OCR can't tell which field a line belongs to):
        [{'box': [x1, y1, x2, y2], 'word': '方泽炎', 'conf': 0.87}, ...]
    box: pixel corners, top-left (x1, y1) and bottom-right (x2, y2).
    conf: Tesseract's confidence scaled to 0-1, the same scale EasyOCR uses.
    """
    data = pytesseract.image_to_data(
        image,
        lang=TESSERACT_LANG,
        config=f"--psm {psm}",
        output_type=pytesseract.Output.DICT,
    )

    # Tesseract reports single words. Group them into lines.
    lines = {}
    for i, word in enumerate(data["text"]):
        if not word.strip():
            continue
        key = (data["block_num"][i], data["par_num"][i], data["line_num"][i])
        x1, y1 = data["left"][i], data["top"][i]
        x2, y2 = x1 + data["width"][i], y1 + data["height"][i]
        if key not in lines:
            lines[key] = {"words": [], "box": [x1, y1, x2, y2], "confs": []}
        line = lines[key]
        line["words"].append(word)
        line["box"] = [
            min(line["box"][0], x1),
            min(line["box"][1], y1),
            max(line["box"][2], x2),
            max(line["box"][3], y2),
        ]
        line["confs"].append(float(data["conf"][i]))

    results = []
    for line in lines.values():
        results.append({
            "box": line["box"],
            "word": _CJK_SPACE.sub("", " ".join(line["words"])),
            "conf": round(sum(line["confs"]) / len(line["confs"]) / 100, 2),
        })
    return results

In [ ]:
ocr_options = {
    "tesseract": ocr_tesseract,
    "easyocr": ocr_easyocr,
}

# Assertion


---
Check the accuracy of the pipeline.


In [ ]:
#TODO

# Pipeline


---
Method to run each pipeline.


In [ ]:
def run_pipeline(image, enhance_fn, channel_fn, ocr_fn):
    start = time.perf_counter()
    x = enhance_fn(image)
    x = channel_fn(x)
    img_text = ocr_fn(x)
    latency = time.perf_counter() - start
    return img_text, latency

# Run all configurations


---
Run all pipeline configurations.


In [ ]:
# This can take a while, only needed once
easy_ocr_reader = easyocr.Reader(['ch_sim'], gpu=False) #KVM@TACC is cpu only

In [ ]:
pipeline_results = []
for e_name, e_fn in enhancement_options.items():
    for c_name, c_fn in channel_options.items():
        for o_name, o_fn in ocr_options.items():
            for item in ds.select(range(10)): #placeholder for speed/ dev only
            # for item in ds: #uncomment for full pipeline run
                
                np_image = np.array(item['image'])
                true_image_result = item['ocr']
                pipeline_image_result, latency = run_pipeline(np_image, e_fn, c_fn, o_fn)
                    # current contract for pipeline_image_result format is a string
                    # if can have in similar format as true_image_results, that would be ideal
                    # easyocr can have detail=0 or 1 so can determine whats best based on tesseract abilities
                # print(pipeline_image_result)
                pipeline_results.append({
                    "target": true_image_result,
                    "result": pipeline_image_result,
                    "latency": latency,
                    "enhancement": e_name,
                    "channel": c_name,
                    "ocr": o_name,
                })
                # probably will want to add more above once we have accuracy metrics
                    
                print(latency)
                

# Display results as table


---
Display #TODO.


In [ ]:
number_of_pipelines = len(enhancement_options) * len(channel_options) * len(ocr_options)
print(f"Number of distinct pipeline configurations: {len(enhancement_options)} enhancement x {len(channel_options)} channel x {len(ocr_options)} OCR = {number_of_pipelines}")